# GrandNexus v7.2.4-rc — the loop that learns, run 5a = recall (pre-registered, logbook 57) — notebook v2
Memory **fixed** at 61 experiences (shipped in the zip, no experience phase). Test = every never-used FOLIO train item whose
story (premises text) is already in the memory: **68 items on 38 stories, sha `4895b2c3fc494d7a`** (yes 26, no 23, unknown 19).
Conditions **none / random / learned**, k = 3, about 200 calls.

**C-recall** (fixed before measuring): paired sign test learned vs none with p < 0.05 in favour of learned, asserted errors not
above none. Secondary: learned > random. Expected: learned > none on about 10 items against 0.

**Why a v2.** The first attempt (2026-11-09 14:07) did not find the release zip where cell 1 looked and silently fell back to an
older GrandNexus zip: the session ran on the run-4 repository, the 5a command stopped at argparse, nothing was measured.
Same protocol, same command, same frozen set; only the loading is hardened: cell 1 takes **only** `GrandNexus_v7.2.4-rc*.zip`
(My Drive/GrandNexus, My Drive, or anywhere 3 levels deep; else upload; any other zip is refused), moves an older
`/content/GrandNexus` aside, and stops unless `git describe` says v7.2.4-rc. Every cell that spends calls re-checks the version.

Run the cells in order. **If a cell prints STOP, do not run the next ones: send the output.** If the Colab assistant offers to
rewrite a cell, decline: the protocol is the result. No item text is printed.

## 1. Get exactly v7.2.4-rc

In [ ]:
# 1. Get EXACTLY GrandNexus v7.2.4-rc (never another zip; stop if it is not found)
import os, glob, fnmatch, subprocess, zipfile, shutil, time
VERSION, ZIP_PATTERN = 'v7.2.4-rc', 'GrandNexus_v7.2.4-rc*.zip'
CONTENT, WORK = '/content', '/content/GrandNexus'
MYDRIVE = '/content/drive/MyDrive'

def gn_version(path):
    """(git describe, has the story-aware bench) of a GrandNexus folder, or (None, False)."""
    if not os.path.isdir(os.path.join(path, '.git')):
        return None, False
    d = subprocess.run('git describe --tags', shell=True, cwd=path, capture_output=True, text=True).stdout.strip()
    bench = os.path.join(path, 'tools', 'bench_learning.py')
    ok = os.path.exists(os.path.join(path, 'tools', 'story_split.py')) and os.path.exists(bench) and '--story-mode' in open(bench, encoding='utf-8').read()
    return d, ok

try:
    from google.colab import drive; drive.mount(os.path.join(CONTENT, 'drive'))
except Exception as e:
    print('Drive not mounted:', e)

# the release zip, newest first: My Drive/GrandNexus and My Drive first, then anywhere in My Drive (3 levels deep);
# any other GrandNexus zip (older versions, session saves) is ignored
found = glob.glob(os.path.join(MYDRIVE, 'GrandNexus', ZIP_PATTERN)) + glob.glob(os.path.join(MYDRIVE, ZIP_PATTERN))
if not found:
    for root, dirs, names in os.walk(MYDRIVE):
        if root[len(MYDRIVE):].count(os.sep) >= 3:
            dirs[:] = []
        found += [os.path.join(root, n) for n in names if fnmatch.fnmatch(n, ZIP_PATTERN)]
found.sort(key=os.path.getmtime)
if found:
    source = found[-1]
else:
    print(f'{ZIP_PATTERN} not found in My Drive: upload it now (any other zip is refused).')
    try:
        from google.colab import files
        up = next(iter(files.upload()))
    except Exception as e:
        raise SystemExit(f'STOP: no {ZIP_PATTERN} on the Drive and no upload possible ({e}).')
    if not fnmatch.fnmatch(os.path.basename(up), ZIP_PATTERN):
        raise SystemExit(f'STOP: {up} is not {ZIP_PATTERN}.')
    source = os.path.join(CONTENT, up)
print('release zip:', source)

# an existing /content/GrandNexus of another version is moved aside, never reused, never deleted
d, ok = gn_version(WORK)
if os.path.exists(WORK) and not (d or '').startswith(VERSION):
    aside = WORK + '_aside_' + time.strftime('%Y%m%d_%H%M%S')
    shutil.move(WORK, aside); print(f'existing {WORK} ({d}) moved aside to {aside}')
if not os.path.exists(WORK):
    z = zipfile.ZipFile(source)
    top = {n.split('/')[0] for n in z.namelist() if n.strip('/')}
    z.extractall(CONTENT if top == {'GrandNexus'} else WORK)
    print('extracted')
subprocess.run('git checkout -q main', shell=True, cwd=WORK)
d, ok = gn_version(WORK)
print('git describe:', d, '| story-aware bench:', ok)
print(subprocess.run('git log --oneline -1 && git branch', shell=True, cwd=WORK, capture_output=True, text=True).stdout)
if not ((d or '').startswith(VERSION) and ok):
    raise SystemExit(f'STOP: {WORK} is {d}, not {VERSION} with the story-aware bench. Do not run the next cells.')
print(f'OK: {VERSION}')


## 2. Packages, key, pre-flight check (1 call)

In [ ]:
# 2. Packages, key, pre-flight check (1 call)
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic', 'clingo', 'sympy', 'networkx', 'psutil'])
import anthropic
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
m = anthropic.Anthropic().messages.create(model='claude-sonnet-5-5', max_tokens=10, messages=[{'role': 'user', 'content': 'Answer only: OK'}])
print('reply:', m.content[0].text, '| id:', m.id, '| model:', m.model)


## 3. Offline tests (0 calls)

In [ ]:
# 3. Offline tests (0 calls)
import subprocess, sys
WORK = '/content/GrandNexus'
for t in ('tests_lab/test_nexus_logic.py', 'tests_lab/test_learning_loop.py', 'tests_lab/test_bench_story.py'):
    r = subprocess.run([sys.executable, '-W', 'ignore', t], cwd=WORK, capture_output=True, text=True)
    last = (r.stdout.strip().splitlines() or ['(no output)'])[-1]
    print(f'{t:40} {last}')
    if 'ALL OK' not in last:
        raise SystemExit(f'STOP: {t} failed (exit {r.returncode}): {r.stderr[-300:]}')


## 4. Pre-registration check (0 calls): 68 items, sha 4895b2c3fc494d7a

In [ ]:
# 4. Pre-registration check (0 calls): the frozen set must be 68 items, sha 4895b2c3fc494d7a
import subprocess
WORK = '/content/GrandNexus'
CMD = 'python tools/bench_learning.py --split train --memory tests_lab/resultats/learning/memory_train_61exp_2026-11-04_191343.json --exp-n 0 --story-mode in-memory --test-n 0 --test-seed 7 --k 3 --conditions none,random,learned --exclude-from "tests_lab/resultats/learning/*.json" "tests_lab/resultats/folio/folio_train_*.json"'
d = subprocess.run('git describe --tags', shell=True, cwd=WORK, capture_output=True, text=True).stdout.strip()
if not d.startswith('v7.2.4-rc'):
    raise SystemExit(f'STOP: {WORK} is {d}, not v7.2.4-rc (re-run cell 1).')
out = subprocess.run(CMD + ' --dry-run', shell=True, cwd=WORK, capture_output=True, text=True)
print(out.stdout, out.stderr[-500:])
if 'test set: 68 items (sha 4895b2c3fc494d7a)' not in out.stdout:
    raise SystemExit('STOP: NOT the pre-registered set. Do not run cell 5; send this output.')
print('pre-registered set confirmed')


## 5. Run 5a (about 200 calls; re-run after an interruption: it resumes)

In [ ]:
# 5. Run 5a (about 200 calls; after an interruption, re-run this cell: it resumes from its checkpoint)
import subprocess, sys
WORK = '/content/GrandNexus'
CMD = 'python tools/bench_learning.py --split train --memory tests_lab/resultats/learning/memory_train_61exp_2026-11-04_191343.json --exp-n 0 --story-mode in-memory --test-n 0 --test-seed 7 --k 3 --conditions none,random,learned --exclude-from "tests_lab/resultats/learning/*.json" "tests_lab/resultats/folio/folio_train_*.json"'
d = subprocess.run('git describe --tags', shell=True, cwd=WORK, capture_output=True, text=True).stdout.strip()
if not d.startswith('v7.2.4-rc'):
    raise SystemExit(f'STOP: {WORK} is {d}, not v7.2.4-rc (re-run cells 1 and 4).')
p = subprocess.Popen(CMD, shell=True, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    sys.stdout.write(line)
p.wait(); print('exit code', p.returncode)


## 6. Save to Drive (commit + complete zip, named run5a / run5a_PARTIAL / run5a_NO_RESULT)

In [ ]:
# 6. Save to Drive (commit + complete zip); the zip name says whether a result exists
import os, glob, subprocess, shutil, time
DRIVE_DIR = '/content/drive/MyDrive/GrandNexus'
WORK = '/content/GrandNexus'
results = sorted(glob.glob(WORK + '/tests_lab/resultats/learning/learning_train_test68_4895b2c3fc494d7a_*.json'))
checkpoints = sorted(glob.glob(WORK + '/tests_lab/resultats/learning/checkpoint_*.json'))
tag = 'run5a' if results else ('run5a_PARTIAL' if checkpoints else 'run5a_NO_RESULT')
for p in glob.glob(WORK + '/**/__pycache__', recursive=True): shutil.rmtree(p, ignore_errors=True)
print(subprocess.run(f'git add -A && git -c user.name=Khaled -c user.email=khaled@grandnexus.local commit -qm "learning loop {tag} (recall)"; '
                     'git describe --tags; git log --oneline -1', shell=True, cwd=WORK, capture_output=True, text=True).stdout)
name = os.path.join(DRIVE_DIR, f'GrandNexus_session_{tag}_' + time.strftime('%Y%m%d_%H%M'))
os.makedirs(DRIVE_DIR, exist_ok=True); shutil.make_archive(name, 'zip', '/content', 'GrandNexus'); print('saved:', name + '.zip')
print('result files:', [os.path.basename(r) for r in results] or 'NONE', '| checkpoints:', [os.path.basename(c) for c in checkpoints] or 'none')
